# MCQ Distractor Generation (DG) — LoRA Training
**Target:** P100 (16GB) or T4 (15GB) &nbsp;|&nbsp; **Model:** `unsloth/Llama-3.2-3B-Instruct`

**Setup:** Upload `dg_train.jsonl` as a Kaggle Dataset, set `DATA_PATH` in Cell 3, run all.


## 1. GPU Detection


In [ ]:
import torch

def detect_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU! Enable GPU in Notebook Settings.")
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    n = torch.cuda.device_count()
    print(f"GPU: {name}  |  VRAM: {vram:.1f} GB  |  Count: {n}  |  CUDA: {torch.version.cuda}")
    for t in ["T4","P100","A100"]:
        if t in name: return t, vram, n
    return "OTHER", vram, n

GPU_TYPE, VRAM_GB, N_GPUS = detect_gpu()


## 2. Install Dependencies


In [ ]:
import subprocess
def _install(cmd, label):
    print(f"[INSTALL] {label} ...", end=" ")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("OK" if r.returncode == 0 else f"FAIL: {r.stderr[-200:]}")

_install("pip install unsloth --upgrade -q", "unsloth")
_install("pip install --upgrade trl transformers peft datasets -q", "trl+transformers+peft+datasets")
_install("pip install rouge-score sentence-transformers scikit-learn structlog numpy -q", "eval deps")

from unsloth import FastLanguageModel
print("\nUnsloth import OK")


## 3. Configuration


In [ ]:
from pathlib import Path

DATA_PATH  = "/kaggle/input/mcq-training-data/dg_train.jsonl"  # ← EDIT THIS
OUTPUT_DIR = "/kaggle/working/mcq_dg"
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"

# DG output is shorter (1 distractor) → larger batch sizes
cfg = {"P100":(8,2),"T4":(4,4),"A100":(16,1)}.get(GPU_TYPE,(4,4))
BATCH_SIZE, GRAD_ACCUM = cfg
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT   = GPU_TYPE != "A100"
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 16, 0.0
EPOCHS, LR, WARMUP, WD = 2, 2e-4, 10, 0.01  # 2 epochs — simpler task
USE_BF16 = torch.cuda.is_bf16_supported()
USE_FP16 = not USE_BF16

print(f"GPU={GPU_TYPE} | batch={BATCH_SIZE} | accum={GRAD_ACCUM} | eff={BATCH_SIZE*GRAD_ACCUM}")
print(f"4bit={LOAD_IN_4BIT} | fp16={USE_FP16} | bf16={USE_BF16} | epochs={EPOCHS}")
assert Path(DATA_PATH).exists(), f"Not found: {DATA_PATH}"


## 4. Prompt & Parser Helpers


In [ ]:
import re

SCORE_CAT_DESC = {"very_weak":"Critical gaps.","weak":"Knows basics; confuses concepts.",
    "moderate":"Partial understanding.","strong":"Proficient; push to ceiling."}
MOD_MAP = {"very_weak":"keep_moderate","weak":"slightly_below_standard","moderate":"standard","strong":"push_to_ceiling"}
MOD_TEXT = {"keep_moderate":"clearly distinguishable","slightly_below_standard":"plausible but not subtle",
    "standard":"matches expected difficulty","push_to_ceiling":"as challenging as possible — subtle"}

def build_dg_chat_prompt(question, correct_answer, question_type, topic, mastery_level, score_category, chunk_text=""):
    mod = MOD_MAP.get(score_category,"standard")
    cat = SCORE_CAT_DESC.get(score_category,"Standard.")
    mt = MOD_TEXT.get(mod, MOD_TEXT["standard"])
    sys = ("You are an expert at creating wrong-but-plausible answer options for educational MCQs.\n\n"
           "OUTPUT FORMAT — output exactly one line and nothing else:\nDISTRACTOR: <the distractor text>\n\nNo JSON, no markdown.")
    ctx = f'\n\nSource content:\n\"\"\"\n{chunk_text[:400]}\n\"\"\"' if chunk_text else ""
    usr = (f"Question: {question}\nCorrect answer: {correct_answer}\nQuestion type: Type {question_type}\n"
           f"Topic: {topic}\nMastery: {mastery_level}\nScore category: {score_category} — {cat}\n"
           f"Difficulty: {mt}{ctx}\n\nGenerate one wrong-but-plausible distractor.")
    return [{"role":"system","content":sys},{"role":"user","content":usr}]

def format_chat(messages, tokenizer):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def extract_dg_output(raw):
    if not raw or not raw.strip(): return None
    t = raw.strip()
    m = re.search(r"DISTRACTOR:\s*(.+)", t, re.DOTALL)
    if m:
        d = m.group(1).strip().split("\n")[0].strip()
        if d: return d
    lines = [l.strip() for l in t.split("\n") if l.strip()]
    if len(lines)==1 and not lines[0].startswith(("QUESTION:","ANSWER:","EXPLANATION:")): return lines[0]
    return None

print("Helpers loaded OK")


## 5. Load & Split Data


In [ ]:
import json
import random
from collections import defaultdict
from sklearn.model_selection import train_test_split

def load_jsonl(path):
    samples = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                try: samples.append(json.loads(line))
                except json.JSONDecodeError: pass
    return samples

def stratified_split_dg(samples, val_ratio=0.1):
    # Shuffle before splitting so sequential book order in the source
    # file does not bias the train/val split or batch composition.
    random.seed(42)
    random.shuffle(samples)
    if len(samples) < 10:
        i = max(1, int(len(samples)*(1-val_ratio))); return samples[:i], samples[i:]
    labels = [f"{s.get('question_type','?')}_{s.get('mastery_level','?')}" for s in samples]
    lc = defaultdict(int)
    for lb in labels: lc[lb] += 1
    labels_clean = [lb if lc[lb]>=2 else "rare" for lb in labels]
    if len(set(labels_clean)) < 2:
        i = max(1, int(len(samples)*(1-val_ratio))); return samples[:i], samples[i:]
    return train_test_split(samples, test_size=val_ratio, stratify=labels_clean, random_state=42)

all_samples = load_jsonl(DATA_PATH)
print(f"Loaded {len(all_samples)} DG samples")
train_samples, val_samples = stratified_split_dg(all_samples)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)}")


## 6. Load Model + Apply LoRA


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=LOAD_IN_4BIT, dtype=None)

model = FastLanguageModel.get_peft_model(model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", use_gradient_checkpointing="unsloth", random_state=42)

tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
tt = sum(p.numel() for p in model.parameters())
print(f"Trainable: {tp:,} / {tt:,} ({100*tp/tt:.3f}%)")


## 7. Build Dataset


In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_samples)
val_dataset = Dataset.from_list(val_samples)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")


## 8. Train


In [ ]:
import time
from trl import SFTTrainer, SFTConfig

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
args = SFTConfig(output_dir=f"{OUTPUT_DIR}/checkpoints", num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, warmup_steps=WARMUP, weight_decay=WD, max_grad_norm=1.0,
    fp16=USE_FP16, bf16=USE_BF16, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=10, save_total_limit=2, report_to="none", seed=42,
    max_seq_length=MAX_SEQ_LENGTH, dataset_text_field="text", packing=False, optim="adamw_8bit")

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_dataset,
    eval_dataset=val_dataset, args=args)

print(f"Starting DG training | eff_batch={BATCH_SIZE*GRAD_ACCUM}")
start = time.time()
train_result = trainer.train()
elapsed = round(time.time() - start, 1)
print(f"Done in {elapsed}s | loss={round(train_result.training_loss,4)}")


## 9. Save LoRA Adapter


In [ ]:
ADAPTER_PATH = f"{OUTPUT_DIR}/adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Saved to: {ADAPTER_PATH}")
for f in sorted(Path(ADAPTER_PATH).iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")


## 10. Evaluate (ROUGE-L + Cosine Similarity)


In [ ]:
import numpy as np
from rouge_score import rouge_scorer as rs_mod
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

FastLanguageModel.for_inference(model)
scorer = rs_mod.RougeScorer(["rougeL"], use_stemmer=True)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
rL = []
mastery_cos = defaultdict(list)
ok = fail = 0
N = min(50, len(val_samples))
print(f"Evaluating on {N} samples...")

for s in val_samples[:N]:
    txt = s.get("text","")
    ca_m = re.search(r"Correct answer:\s*(.+?)(?:\n|$)", txt)
    ref_m = re.search(r"DISTRACTOR:\s*(.+?)(?:\n|$)", txt)
    ca = ca_m.group(1).strip() if ca_m else ""
    ref = ref_m.group(1).strip() if ref_m else ""
    mastery = s.get("mastery_level","unknown")

    msgs = build_dg_chat_prompt("",ca,s.get("question_type","4a"),"",mastery,s.get("score_category","moderate"))
    inp = tokenizer(format_chat(msgs,tokenizer), return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    inp = {k:v.to(model.device) for k,v in inp.items()}
    ilen = inp["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=80, temperature=0.0, do_sample=False)
    raw = tokenizer.decode(out[0][ilen:], skip_special_tokens=True)
    p = extract_dg_output(raw)
    if p:
        ok += 1
        if ref: rL.append(scorer.score(ref,p)["rougeL"].fmeasure)
        if ca:
            embs = embedder.encode([ca,p], convert_to_numpy=True, show_progress_bar=False)
            mastery_cos[mastery].append(float(cosine_similarity([embs[0]],[embs[1]])[0][0]))
    else: fail += 1

avg = lambda l: round(float(np.mean(l)),4) if l else 0.0
print(f"\nROUGE-L={avg(rL)} | Parse={ok}/{N}")
print("Cosine sim per mastery:")
for m,v in sorted(mastery_cos.items()): print(f"  {m}: {avg(v)}")


## 11. Save Metrics


In [ ]:
metrics = {"task":"DG","gpu":GPU_TYPE,"base_model":BASE_MODEL,"epochs":EPOCHS,
    "batch_size":BATCH_SIZE,"lora_r":LORA_R,"train_loss":round(train_result.training_loss,4),
    "training_time_s":elapsed,"rougeL":avg(rL),"parse_ok":ok,"parse_fail":fail,
    "train_n":len(train_samples),"val_n":len(val_samples)}
mp = f"{OUTPUT_DIR}/training_metrics.json"
with open(mp,"w") as f: json.dump(metrics,f,indent=2)
print(f"Metrics: {mp}\nDone! Download /kaggle/working/mcq_dg/ from Output tab.")
